# Preprocessing

In [1]:
import pandas as pd
import os

# Configuration -
CONFIG = {
    "input_folder": "01_Raw Data",
    "input_file": "Stromerzeugung.xlsx",    
    "output_folder": "02_Preprocessing",
    "csv_filename": "Gesamterzeugung_hourly.csv"
}


df = pd.read_excel(os.path.join(CONFIG['input_folder'], CONFIG['input_file']))

print(f"Loaded: {df.shape}")



Loaded: (140256, 13)


In [2]:
# Remove [MWh] from all column names
df.columns = df.columns.str.replace(' \[MWh\]', '', regex=True).str.strip()

columns = list(range(len(df.columns)))# all column indices from df
#columns = [0,1,2,3]

# Checking classes  of the columns
for col_idx in columns:
    col_name = df.columns[col_idx]
    print(f"Column {col_name:<40} {df[col_name].apply(type).unique()}")
   





Column Datum von                                [<class 'str'>]
Column Biomasse                                 [<class 'float'>]
Column Wasserkraft                              [<class 'float'>]
Column Wind Offshore                            [<class 'float'>]
Column Wind Onshore                             [<class 'float'>]
Column Photovoltaik                             [<class 'float'>]
Column Sonstige Erneuerbare                     [<class 'float'>]
Column Kernenergie                              [<class 'float'>]
Column Braunkohle                               [<class 'float'>]
Column Steinkohle                               [<class 'float'>]
Column Erdgas                                   [<class 'float'>]
Column Pumpspeicher                             [<class 'float'>]
Column Sonstige Konventionelle                  [<class 'float'>]


In [3]:
# Convert columns to the correct format
columns.remove(0)

for col_idx in columns:
    if col_idx < len(df.columns):
        col_name = df.columns[col_idx]
        #print(f"Fixing: {col_name}")
        df[col_name] = pd.to_numeric(df[col_name], errors='coerce')



print("All columns are now numeric!")


All columns are now numeric!


In [4]:
display(df)

,Datum von,Biomasse,Wasserkraft,Wind Offshore,Wind Onshore,Photovoltaik,Sonstige Erneuerbare,Kernenergie,Braunkohle,Steinkohle,Erdgas,Pumpspeicher,Sonstige Konventionelle
0,01.01.2015 00:00,1005.50,288.25,130.00,2028.25,0.00,33.25,2685.50,3964.75,805.50,317.25,391.00,1235.00
1,01.01.2015 00:15,1007.00,287.75,129.25,2023.00,0.00,33.25,2646.25,3950.75,833.00,316.25,373.25,1213.75
2,01.01.2015 00:30,1006.50,292.75,128.50,2040.25,0.00,33.25,2660.75,3912.25,806.25,313.50,426.50,1218.50
3,01.01.2015 00:45,1005.25,289.50,128.75,2036.50,0.00,33.25,2718.00,3859.50,775.00,279.25,335.00,1242.00
4,01.01.2015 01:00,999.00,295.25,128.75,2045.75,0.00,33.25,2772.25,3888.00,633.50,261.50,247.50,1247.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...
140251,31.12.2018 22:45,1203.50,376.00,838.75,4362.25,0.25,32.50,2243.00,2373.00,892.50,987.75,21.50,431.25
140252,31.12.2018 23:00,1198.00,380.50,836.50,4448.75,0.25,32.50,2250.50,2289.75,906.75,938.75,113.50,433.00
140253,31.12.2018 23:15,1200.00,381.75,760.75,4545.00,0.25,33.50,2252.50,2240.75,906.75,936.25,92.25,432.75
140254,31.12.2018 23:30,1197.75,387.00,720.75,4651.50,0.25,33.50,2262.50,2153.75,910.75,933.50,141.00,432.50


In [5]:
#New column 'Stromerzeugung Gesamt' as sum of columns 1 to 12
df['Stromerzeugung Gesamt'] = df.iloc[:, 1:13].sum(axis=1)
# Rename first column to 'Datum'
df.rename(columns={df.columns[0]: 'Datum'}, inplace=True)

# Convert 'Datum' column to datetime with day first format
df['Datum'] = pd.to_datetime(df['Datum'], dayfirst=True)

neu = df[['Datum', df.columns[13]]].copy()
neu = neu.set_index('Datum')

neu = neu.resample('h').sum()
display(neu)


,Stromerzeugung Gesamt
Datum,
2015-01-01 00:00:00,51238.75
2015-01-01 01:00:00,49749.00
2015-01-01 02:00:00,49014.75
2015-01-01 03:00:00,47954.75
2015-01-01 04:00:00,48187.00
...,...
2018-12-31 19:00:00,55441.00
2018-12-31 20:00:00,54499.50
2018-12-31 21:00:00,54526.75


In [6]:
#Save
neu.to_csv(os.path.join(CONFIG['output_folder'], CONFIG['csv_filename']), index=True)

In [7]:
# Checking if it was saved correctly
neu2 = pd.read_csv(os.path.join(CONFIG['output_folder'], CONFIG['csv_filename']), index_col='Datum', parse_dates=True)

display(neu2)

,Stromerzeugung Gesamt
Datum,
2015-01-01 00:00:00,51238.75
2015-01-01 01:00:00,49749.00
2015-01-01 02:00:00,49014.75
2015-01-01 03:00:00,47954.75
2015-01-01 04:00:00,48187.00
...,...
2018-12-31 19:00:00,55441.00
2018-12-31 20:00:00,54499.50
2018-12-31 21:00:00,54526.75
